In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import sys
import os
%pip install -q faiss-gpu-cu12
%pip install -q duckdb polars

sys.path.append(os.path.abspath('/content/drive/My Drive/Thesis/book_recsys'))
sys.path.insert(0,'/content/drive/My Drive/Thesis/book_recsys/app')

from app.recommenders.vector_recommender import VectorRecommender
from app.recommenders.processer import BookProcessor, UserProcessor
from app.recommenders.hybrid_recommend import HybridRecommender

In [4]:
_base_dir = '/content/drive/My Drive/Thesis/book_recsys'
_data_dir = f'{_base_dir}/data/processed_2'
_models_dir = f'{_data_dir}/model_v1'

user_index = f'{_models_dir}/user_index.csv'
imtem_index = f'{_models_dir}/cb_sbert_book_index.csv'
history_db = f'{_models_dir}/user_history.db'

sbert_item_matrix = f'{_models_dir}/cb_sbert_matrix.npy'
sbert_user_matrix = f'{_models_dir}/cb_sbert_user_matrix.npy'
sbert_item_faiss_index = f'{_models_dir}/cb_sbert_matrix.index'

ials_user_matrix = f'{_models_dir}/cf_als_user_profiles.npy'
ials_item_matrix = f'{_models_dir}/cf_als_item_matrix.npy'
ials_item_faiss_index = f'{_models_dir}/cf_als_item_matrix.index'

test_file = f'{_data_dir}/test_pos.csv'

In [7]:
book_proc = BookProcessor(item_index_path= imtem_index)
user_proc = UserProcessor(user_index_path= user_index,history_db_path= history_db)

Loading Book mapping from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/cb_sbert_book_index.csv ...
 -> Loaded 1,173,713 books into RAM dictionary.
Loading User mapping from /content/drive/My Drive/Thesis/book_recsys/data/processed_2/model_v1/user_index.csv...
Connecting to Persistent User History DB...


In [9]:
cb_recommend = VectorRecommender(
    book_processor= book_proc,
    user_processor= user_proc,
    faiss_index_path=sbert_item_faiss_index,
    user_vectors_path = sbert_user_matrix,
    use_gpu = False
)

Loading Data & CPU FAISS Index...
Recommender Engine Ready (Users: 743,401)


In [10]:
cf_recommend = VectorRecommender(
    book_processor= book_proc,
    user_processor= user_proc,
    faiss_index_path=ials_item_faiss_index,
    user_vectors_path = ials_user_matrix,
    use_gpu = False
)

Loading Data & CPU FAISS Index...
Recommender Engine Ready (Users: 743,401)


In [19]:
import gc
gc.collect()

hybrid_recommender = HybridRecommender(
    cf_engine= cf_recommend,
    cbf_engine= cb_recommend,
    weight_cf = 0.8,
    weight_cbf =0.2
)

Hybrid Engine Ready! (Weights: CF=0.80, CBF=0.20)


In [21]:
hybrid_recommender.recommend("08b90f5940562417d216973ba3093544")

[RecommendItem(item_id='22219682', score=0.8, rank=1),
 RecommendItem(item_id='2287468', score=0.7615300756042461, rank=2),
 RecommendItem(item_id='6993490', score=0.3977286382521945, rank=3),
 RecommendItem(item_id='15803047', score=0.3854368315089609, rank=4),
 RecommendItem(item_id='1316', score=0.291017101422474, rank=5),
 RecommendItem(item_id='30841984', score=0.2516899560012131, rank=6),
 RecommendItem(item_id='381817', score=0.2130513879804127, rank=7),
 RecommendItem(item_id='643750', score=0.20896436756547787, rank=8),
 RecommendItem(item_id='18219909', score=0.2, rank=9),
 RecommendItem(item_id='9647295', score=0.18768559074925792, rank=10)]

In [22]:
cb_recommend.recommend_realtime("08b90f5940562417d216973ba3093544")

[RecommendItem(item_id='18219909', score=0.8528110980987549, rank=1),
 RecommendItem(item_id='13064460', score=0.8190400004386902, rank=2),
 RecommendItem(item_id='26155235', score=0.8185950517654419, rank=3),
 RecommendItem(item_id='23876302', score=0.8064031600952148, rank=4),
 RecommendItem(item_id='10350530', score=0.8018452525138855, rank=5),
 RecommendItem(item_id='15861254', score=0.8005479574203491, rank=6),
 RecommendItem(item_id='15775388', score=0.7991534471511841, rank=7),
 RecommendItem(item_id='23124925', score=0.7974194884300232, rank=8),
 RecommendItem(item_id='8449885', score=0.7957225441932678, rank=9),
 RecommendItem(item_id='15791200', score=0.7956038117408752, rank=10)]

In [23]:
cf_recommend.recommend_realtime("08b90f5940562417d216973ba3093544")

[RecommendItem(item_id='22219682', score=1.2379283905029297, rank=1),
 RecommendItem(item_id='2287468', score=1.2228633165359497, rank=2),
 RecommendItem(item_id='6993490', score=1.080396294593811, rank=3),
 RecommendItem(item_id='15803047', score=1.07558274269104, rank=4),
 RecommendItem(item_id='1316', score=1.0386073589324951, rank=5),
 RecommendItem(item_id='30841984', score=1.0232065916061401, rank=6),
 RecommendItem(item_id='381817', score=1.008075475692749, rank=7),
 RecommendItem(item_id='643750', score=1.0064749717712402, rank=8),
 RecommendItem(item_id='9647295', score=0.9981420636177063, rank=9),
 RecommendItem(item_id='2445973', score=0.988298237323761, rank=10)]

In [ ]:
import json
import pandas as pd
import concurrent.futures

def save_candidates_to_json(predictions_dict, output_path):
    print(f"\n💾 Đang ghi {len(predictions_dict):,} users ra đĩa cứng: {output_path}...")
    json_ready_data = {
        str(uid): [item.to_api_dict() for item in rec_items]
        for uid, rec_items in predictions_dict.items()
    }
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(json_ready_data, f, ensure_ascii=False, indent=2)
    print(" -> Ghi file thành công!")

def get_chunks(lst, chunk_size):
    """Hàm cắt lô dữ liệu"""
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

def process_chunk(engine, chunk_users, k=100):
    """Hàm Wrapper để ném vào ThreadPool"""
    # Hàm này sẽ gọi recommend_batch (Có kết hợp cả CPU Prep, GPU Search, CPU Post)
    return engine.recommend_batch(chunk_users, k=k, nprobe=50)

def run_model_pipeline(engine, valid_users, chunk_size, output_file, max_workers=3):
    """Khởi chạy kiến trúc Pipeline bằng ThreadPool"""
    all_preds = {}
    chunks = list(get_chunks(valid_users, chunk_size))
    total_chunks = len(chunks)

    print(f"🚀 Bắt đầu xử lý {total_chunks} Chunks với {max_workers} Luồng song song...")

    # Sử dụng ThreadPoolExecutor để chạy đa luồng
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit toàn bộ tác vụ vào Pool
        future_to_chunk = {
            executor.submit(process_chunk, engine, chunk): i
            for i, chunk in enumerate(chunks, 1)
        }

        # Lấy kết quả ngay khi có một Luồng chạy xong (Asynchronous)
        for future in concurrent.futures.as_completed(future_to_chunk):
            chunk_id = future_to_chunk[future]
            try:
                chunk_result = future.result()
                all_preds.update(chunk_result)
                print(f" -> Hoàn thành Chunk {chunk_id}/{total_chunks} ({len(chunk_result)} users)", end="\r")
            except Exception as exc:
                print(f"\n❌ Chunk {chunk_id} sinh ra lỗi: {exc}")

    save_candidates_to_json(all_preds, output_file)


def main():
    print("🌟 KHỞI ĐỘNG HỆ THỐNG ASYNC PIPELINE 🌟")

    user_proc = UserProcessor(
        user_index_path="output/user_index.csv",
        interactions_path="data/train_interactions.parquet"
    )
    book_proc = BookProcessor("output/item_index.csv")

    valid_df = pd.read_csv("data/valid_interactions.csv")
    valid_users = valid_df['user_id'].unique().tolist()

    # Cấu hình siêu tham số hệ thống
    CHUNK_SIZE = 5000     # Giữ an toàn cho VRAM (2-3GB)
    MAX_WORKERS = 3       # Tối ưu: 1 luồng Prep, 1 luồng GPU, 1 luồng Post-process

    # ==========================================
    # 1. Pipeline cho CF
    # ==========================================
    print("\n" + "="*40 + "\n[1] CHẠY MÔ HÌNH COLLABORATIVE FILTERING")
    cf_engine = UniversalVectorRecommender(
        user_proc=user_proc, book_proc=book_proc,
        user_vectors_path="output/cf_user_vectors.npy",
        faiss_index_path="output/cf_item_ivf.index",
        use_gpu=True
    )

    run_model_pipeline(cf_engine, valid_users, CHUNK_SIZE, "output/cf_valid_candidates.json", MAX_WORKERS)
    del cf_engine # Dọn RAM/VRAM

    # ==========================================
    # 2. Pipeline cho CBF
    # ==========================================
    print("\n" + "="*40 + "\n[2] CHẠY MÔ HÌNH CONTENT-BASED FILTERING")
    cbf_engine = UniversalVectorRecommender(
        user_proc=user_proc, book_proc=book_proc,
        user_vectors_path="output/cbf_user_profiles.npy",
        faiss_index_path="output/cbf_item_ivf.index",
        use_gpu=True
    )

    run_model_pipeline(cbf_engine, valid_users, CHUNK_SIZE, "output/cbf_valid_candidates.json", MAX_WORKERS)

if __name__ == "__main__":
    main()